In [1]:
import getpass
import os
def _set_if_undefined(var: str) -> None:
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)


_set_if_undefined("OPENAI_API_KEY")
#sk-afda581a0d314c64b897b4df1b8bdc05,这里为了复制方便，实际开发不使用明码
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:

import sentence_transformers  # ★ 必须先导入，避免 Windows 上 DLL 加载顺序导致 segfault
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike
import chromadb
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore


In [3]:
CHUNK_SIZE=256
CHUNK_OVERLAP=0
TOP_K = 5
EMBEDDING_MODEL_NAME="BAAI/bge-base-zh-v1.5"

In [4]:

# ---- 1. 全局配置：embedding / LLM / 切分器 ----
# Settings 是 LlamaIndex 的全局默认，配一次后面所有步骤都用它
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL_NAME)
Settings.llm = OpenAILike(
    model="deepseek-v4-pro",
    api_base="https://api.deepseek.com",   # 注意带 /v1
    is_chat_model=True,        # ★ 必加,否则会去打 completions 口然后报错
    context_window=128000,
    max_tokens=1024*8,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:

Settings.node_parser = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

In [6]:

# ---- 2. 加载 PDF（每页通常是一个 Document）----
documents = SimpleDirectoryReader("./data").load_data()
print(repr(documents[0].text[:300]))   # ★ 先看这个

'广州大学研究生学位论文中期报告表\n研 究 生\n姓 名 王克 所在\n学院\n计算机科学与\n网络工程学院\n导\n师 朱恩强 学\n号 2112406169\n专 业 软件工程 研究方向 人工智能\n论文题目：通过有效性感知 Transformer 解决约束满足问题\n论文工作中期报告时间：2026.5.25 地点：A3-308\n参加论文中期报告教师（要求副高以上职称的教师不少于三人）签名：\n论文工作情况（包括已取得的阶段性结果和存在的问题，下一步的工作计划等）：\n本项目自启动以来进展顺利，已圆满完成核心算法设计、全量对比实验及底层机\n制解析，取得了一系列突破性的阶段成果。我们创新性地提出了一种融合关系结构偏\n'


In [7]:
print(f"loaded {len(documents)} document pages")

# ---- 3. 连接 Chroma（持久化到本地 ./chroma_db）----
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection("my_rag_chunk_size_256")  # 换名避免与旧 512-d collection 冲突
vector_store = ChromaVectorStore(chroma_collection=collection)#把 Chroma 的 collection 包装成 LlamaIndex 认识的"转接头"
storage_context = StorageContext.from_defaults(vector_store=vector_store)#StorageContext—— LlamaIndex 用来回答"东西到底存哪儿"的对象,

# ---- 4. 建索引：切分 → 向量化 → 写入 Chroma，这一步一气呵成 ----
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)

# ---- 5. 检索 top-5 + Claude 生成 ----
query_engine = index.as_query_engine(similarity_top_k=TOP_K)
response = query_engine.query("这份中期报告表的研究生是谁")
print("\n=== 回答 ===")
print(response)

# 调试：看看到底检索到了哪 5 个片段
for i, node in enumerate(response.source_nodes):
    print(f"\n--- chunk {i}  (score={node.score:.3f}) ---")
    print(node.text[:200])

loaded 9 document pages


2026-06-25 16:15:37,314 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"



=== 回答 ===
王克

--- chunk 0  (score=0.390) ---
广州大学研究生学位论文中期报告表
研 究 生
姓 名 王克 所在
学院
计算机科学与
网络工程学院
导
师 朱恩强 学
号 2112406169
专 业 软件工程 研究方向 人工智能
论文题目：通过有效性感知 Transformer 解决约束满足问题
论文工作中期报告时间：2026.5.

--- chunk 1  (score=0.374) ---
目前学位论文
的框架已基本成型，只需在翻译润色和理论分析上进一步深化。鉴于该生目前的良好进展，同
意通过中期检查，建议按计划推进后续工作。
导师签名：
年 月 日

--- chunk 2  (score=0.374) ---
目前学位论文
的框架已基本成型，只需在翻译润色和理论分析上进一步深化。鉴于该生目前的良好进展，同
意通过中期检查，建议按计划推进后续工作。
导师签名：
年 月 日

--- chunk 3  (score=0.374) ---
目前学位论文
的框架已基本成型，只需在翻译润色和理论分析上进一步深化。鉴于该生目前的良好进展，同
意通过中期检查，建议按计划推进后续工作。
导师签名：
年 月 日

--- chunk 4  (score=0.367) ---
1
广 州 大 学
研 究 生 中 期 考 核 表
培养层次： □博士 ☑硕士
学位类型： □学术型 ☑专业型
姓 名： 王克
学 院： 计算机科学与网络工程学院
年 级： 2024 级
专业/领域： 软件工程
校内指导教师： 朱恩强
校外指导教师： 无
广州大学研究生院
2026年 5 月 22 日 填
学号 2112406169


In [8]:
QA_PAIRS = [
    # ===== 直接查找型（12 个）：答案在文档里明确写着 =====
    {"question": "这份材料的研究生姓名是什么？", "reference": "王克"},
    {"question": "王克的校内指导教师是谁？", "reference": "朱恩强"},
    {"question": "王克的学号是多少？", "reference": "2112406169"},
    {"question": "学位论文的题目是什么？", "reference": "通过有效性感知 Transformer 解决约束满足问题"},
    {"question": "王克所在的学院是哪个？", "reference": "计算机科学与网络工程学院"},
    {"question": "王克的专业是什么？", "reference": "软件工程"},
    {"question": "中期考核答辩小组的组长是谁？", "reference": "强小利"},
    {"question": "该模型在 9x9 数独基准上达到的求解精度是多少？", "reference": "99.94%"},
    {"question": "王克把英文论文投稿到了哪个学术会议？", "reference": "NeurIPS 2026"},
    {"question": "投稿的英文论文有多少页？", "reference": "45 页"},
    {"question": "王克是哪一年级的研究生？", "reference": "2024 级"},
    {"question": "学位论文开题报告的通过时间是什么时候？", "reference": "2025年12月31日"},

    # ===== 需要综合型（5 个）：要跨段落/跨文档聚合或筛选 =====
    {"question": "中期考核答辩小组由哪几位成员组成？",
     "reference": "共4位：组长强小利、成员陈智华和许鹏、记录员孙思"},
    {"question": "该研究的模型成功迁移到了哪些组合优化问题上？",
     "reference": "16x16 复杂数独、10x10 二元谜题、Erdős-Rényi 随机图着色"},
    {"question": "王克的研究目前面临哪些主要挑战或局限？",
     "reference": "辅助约束损失与主目标产生梯度冲突；模型高度依赖人工预定义的离散关系矩阵，尚不能从非结构化场景自动提取关系图"},
    {"question": "王克达到毕业要求的后续计划是什么？",
     "reference": "将45页英文论文翻译成中文并少量扩充润色，预计研三第一学期完成学位论文"},
    {"question": "论文中期报告与中期考核的答辩日期和地点分别是？两者是否一致？",
     "reference": "均为2026年5月25日、地点A3-308，两者一致"},

    # ===== 诚实性探针（3 个）：文档里没有答案，正确表现是“说不知道”而非硬编 =====
    {"question": "王克的本科毕业院校是哪所大学？", "reference": "文档中未提及"},
    {"question": "这篇论文除王克外的合作作者有哪些？", "reference": "文档中未提及"},
    {"question": "王克的联系邮箱或电话是多少？", "reference": "文档中未提及"},
]

In [9]:


import os
import nest_asyncio  # Jupyter 里跑需要它；纯脚本下加着也无害
nest_asyncio.apply()


query_engine = index.as_query_engine(similarity_top_k=TOP_K)



# ============ C. 跑 RAG，收集每个问题的 答案 + 检索片段 ============
def run_rag(question: str):
    """返回 (答案文本, 检索到的片段列表)"""
    resp = query_engine.query(question)
    answer = str(resp)
    contexts = [n.node.get_content() for n in resp.source_nodes]
    return answer, contexts


records = []
for i, qa in enumerate(QA_PAIRS):
    ans, ctxs = run_rag(qa["question"])
    records.append({
        "user_input": qa["question"],      # RAGAS 字段名：问题
        "retrieved_contexts": ctxs,         # RAGAS 字段名：检索片段（list）
        "response": ans,                    # RAGAS 字段名：RAG 给出的答案
        "reference": qa["reference"],       # RAGAS 字段名：标准答案（旧版叫 ground_truth）
    })
    print(f"[{i+1}/{len(QA_PAIRS)}] {qa['question']}  ->  {ans[:40]}...")


2026-06-25 16:15:41,080 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


[1/20] 这份材料的研究生姓名是什么？  ->  根据提供的材料内容，没有提及研究生的具体姓名，因此无法得知。...


2026-06-25 16:15:45,977 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


[2/20] 王克的校内指导教师是谁？  ->  王克的校内指导教师是朱恩强。...


2026-06-25 16:15:48,680 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


...

[18/20] 王克的本科毕业院校是哪所大学？  ->  根据现有材料，无法得知王克的本科毕业院校。...


2026-06-25 16:17:04,656 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


[19/20] 这篇论文除王克外的合作作者有哪些？  ->  根据提供的资料，文中仅提及研究生王克及其导师朱恩强，未列出其他合作作者。...


2026-06-25 16:17:08,357 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"


[20/20] 王克的联系邮箱或电话是多少？  ->  很抱歉，根据所提供的文档内容，无法找到王克的联系邮箱或电话。文档中包含其姓名、学...


In [10]:

# ============ D. 构建评审模型（DeepSeek 当 judge + 本地 bge 当 embedding）============
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="deepseek-v4-pro",
    base_url="https://api.deepseek.com/",
    temperature=0,   # ★ judge 必须 temperature=0，否则分数不可复现
))
evaluator_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
)


# ============ E. 跑评估 ============
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

evaluation_dataset = EvaluationDataset.from_list(records)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_emb,
    run_config=RunConfig(max_workers=4, timeout=120),  # DeepSeek 限流时把 max_workers 调小
)

print("\n========== 基准分数（均值）==========")
print(result)

# 存档：聚合分 + 每条样本明细，连同关键配置一起记录，才有对照价值
df = result.to_pandas()
df.to_csv("ragas_baseline_per_sample.csv", index=False)

with open("ragas_baseline_summary.txt", "a", encoding="utf-8") as f:
    f.write("=== RAGAS Baseline ===\n")
    f.write(f"样本数: {len(records)}\n")
    f.write("配置（必须记录，否则没法对照）:\n")
    f.write(f"  embedding = {EMBEDDING_MODEL_NAME}\n")
    f.write("  generator = deepseek-v4-pro\n")
    f.write("  judge     = deepseek-v4-pro (temperature=0)\n")
    f.write(f"  chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}, top_k={TOP_K}\n\n")
    f.write(str(result) + "\n")

print("\n已保存：ragas_baseline_per_sample.csv（逐条）/ ragas_baseline_summary.txt（汇总+配置）")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_31304\1836250146.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(
2026-06-25 16:17:13,108 - INFO - No device provided, using cpu
2026-06-25 16:17:13,748 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/modules.json "HTTP/1.1 308 Permanent Redirect"
2026-06-25 16:17:13,904 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 308 Permanent Redirect"
2026-06-25 16:17:13,906 - INFO - Loading SentenceTransformer model from BAAI/bge-base-zh-v1.5.
2026-06-25 16:17:14,053 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/config_sentence_transformers.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-06-25 16:17:15,394 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/processor_config.json "HTTP/1.1 308 Permanent Redirect"
2026-06-25 16:17:15,549 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 308 Permanent Redirect"
2026-06-25 16:17:15,682 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 308 Permanent Redirect"
2026-06-25 16:17:15,874 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 308 Permanent Redirect"
2026-06-25 16:17:16,026 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 308 Permanent Redirect"
2026-06-25 16:17:16,188 - INFO - HTTP Request: HEAD https://hf-mirror.com/BAAI/bge-base-zh-v1.5/resolve/main/config.json "HTTP/1.1 308 Permanent Redirect"
2026-

Evaluating:   0%|          | 0/80 [00:00<?, ?it/s]

2026-06-25 16:17:25,191 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-06-25 16:17:25,198 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-06-25 16:17:25,573 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-06-25 16:17:25,575 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-06-25 16:17:31,701 - WARNING - LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
2026-06-25 16:17:35,772 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 200 OK"
2026-06-25 16:17:37,785 - INFO - HTTP Request: POST https://api.deepseek.com/chat/completions "HTTP/1.1 400 Bad Request"
2026-06-25 16:17:37,787 - ERROR - Exception raised in Job[5]: BadRequestError(Error code: 400 - {'error': {'message': 'Invalid n value (currently only n = 1 is supported)', 'type': 'invalid_request_err


========== 基准分数（均值）==========
{'faithfulness': 0.9419, 'answer_relevancy': 0.0000, 'context_precision': 0.6300, 'context_recall': 0.6250}

已保存：ragas_baseline_per_sample.csv（逐条）/ ragas_baseline_summary.txt（汇总+配置）
